# blocking-test-check
Generated from `code/business_entity_resolution/src/run_blocking_test_check.py` by `jobs/build_notebook.py`. Edit the script, not this notebook.

In [ ]:
!pip install -q polars==1.44.2 rapidfuzz==3.14.6 anyascii==0.3.3 scipy==1.18.1 scikit-learn==1.9.1 sparse-dot-topn==1.2.0

In [ ]:
import os
os.makedirs('/tmp/src', exist_ok=True)
for root, dirs, files in os.walk('/kaggle/input'):
    print(root, dirs, files)

In [ ]:
%%writefile /tmp/src/blocking.py
"""Candidate generation (blocking): union of several searches, per country.

Methods (all per country; the country is a compute split, 0 true pairs cross countries):
- name       forward TF-IDF top-k on core_name
- name_city  forward TF-IDF top-k on core_name + city
- name_addr  forward TF-IDF top-k on core_name + normalised address
- reverse    for every S2/S3 record, its top-m S1 on core_name + address (all S1 of the split compete); the
             record's best score is kept so a margin rule (keep rank 2 only if close to rank 1) can be applied
- rare       S2/S3 records sharing a rare core_name token (document frequency <= RARE_DF in the country)

TF-IDF: char_wb 3-4-grams, fitted per country on train + test records (no labels; a seeded sample
of up to FIT_SAMPLE texts). N-grams in more than MAX_DF of the texts are dropped: they carry almost no
weight and dominate the cost of the sparse product.

Search scope (a compute split, not a filter):
- "state": a forward query searches the pool records of its own state plus all pool records without a
  state; an S1 without a state searches the whole country. The reverse search runs within states; a pool
  record without a state searches all S1 of the country (fallback). Before bucketing, a missing state is
  inferred from the city, then the dept (state_maps / fill_states, learned from records that have both).
- "country": every search covers the whole country (affordable on a GPU); recovers pairs whose states
  disagree.
The rare-token index ignores states in both scopes.

Top-k search runs on the GPU when torch sees one (dense score blocks + torch.topk), otherwise on CPU with
sparse_dot_topn. check_gpu() compares the two on a slice before a run (tie-aware: scores per rank and
exact recomputation) and the faster device does the searches.

Near-identical decoys are NOT filtered here; every method keeps its full top-k.
"""
from __future__ import annotations

import time
from collections.abc import Callable

import numpy as np
import polars as pl
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn

SEED = 42
FIT_SAMPLE = 3_000_000
MAX_DF = 0.02
RARE_DF = 20
NGRAMS = (3, 4)
FORWARD = ("name", "name_city", "name_addr")
REVERSE_REP = "name_addr"
NONE = "__none__"
ALL = "__all__"
SCOPES = ("state", "country")
INFER_MIN_N, INFER_MIN_SHARE = 20, 0.9  # state inference from city / dept (bucket placement only)
N_THREADS = 4


def texts(df: pl.DataFrame) -> pl.DataFrame:
    """Text representations used by the searches (all from norm-v3 columns)."""
    core = pl.col("core_name")
    return df.select(
        core.alias("name"),
        pl.concat_str([core, pl.col("city").fill_null("")], separator=" ").str.strip_chars().alias("name_city"),
        pl.concat_str([core, pl.col("addr_norm").str.replace_all(",", "")], separator=" ").str.strip_chars().alias("name_addr"),
    )


def fit_vectorizer(corpus: pl.Series) -> TfidfVectorizer:
    if corpus.len() > FIT_SAMPLE:
        corpus = corpus.sample(FIT_SAMPLE, seed=SEED)
    v = TfidfVectorizer(analyzer="char_wb", ngram_range=NGRAMS, min_df=2, max_df=MAX_DF,
                        sublinear_tf=True, dtype=np.float32)
    v.fit(corpus.to_list())
    return v


try:  # GPU path (Kaggle GPU); falls back to sparse_dot_topn on CPU
    import torch
    USE_GPU = torch.cuda.is_available()
except ImportError:
    torch, USE_GPU = None, False
GPU_BLOCK_BYTES = 2 * 1024**3  # cap for each dense block per step (scores: chunk x pool; queries: chunk x vocab)


def gpu_info() -> str:
    if not USE_GPU:
        return "no GPU (CPU sparse_dot_topn)"
    return f"torch {torch.__version__}, {torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GB"


def _to_torch_csr(M: sparse.csr_matrix, device: str):
    return torch.sparse_csr_tensor(torch.from_numpy(M.indptr.astype(np.int32)), torch.from_numpy(M.indices.astype(np.int32)),
                                   torch.from_numpy(M.data.astype(np.float32)), size=M.shape, device=device)


def _topn_gpu(Q: sparse.csr_matrix, P: sparse.csr_matrix, k: int, keep_col: np.ndarray | None = None, device: str = "cuda"):
    """Top-k of Q @ P.T per row on the GPU: P stays on the card as CSR; each query chunk is densified,
    multiplied (cuSPARSE SpMM) and torch.topk keeps the best k pool rows per query."""
    Pg = _to_torch_csr(P, device)
    chunk = int(max(1, min(4096, GPU_BLOCK_BYTES // (4 * P.shape[0]), GPU_BLOCK_BYTES // (4 * Q.shape[1]))))
    rows, cols, vals, ranks = [], [], [], []
    for s in range(0, Q.shape[0], chunk):
        Qc = _to_torch_csr(Q[s:s + chunk], device).to_dense()      # chunk x V
        scores = torch.sparse.mm(Pg, Qc.T).T.contiguous()           # chunk x pool
        v, i = torch.topk(scores, k, dim=1)                         # chunk x k, sorted descending
        v, i = v.cpu().numpy().reshape(-1), i.cpu().numpy().reshape(-1).astype(np.int64)
        n = scores.shape[0]
        r, rk = np.repeat(np.arange(s, s + n), k), np.tile(np.arange(1, k + 1), n)
        keep = v > 0  # as on CPU: only pairs sharing an n-gram
        if keep_col is not None:
            keep &= keep_col[i]
        rows.append(r[keep]); cols.append(i[keep]); vals.append(v[keep]); ranks.append(rk[keep])
        del Qc, scores
    del Pg
    return tuple(np.concatenate(x) for x in (rows, cols, vals, ranks))


def _topn_cpu(Q: sparse.csr_matrix, P: sparse.csr_matrix, k: int, keep_col: np.ndarray | None = None):
    R = sp_matmul_topn(Q, P.T.tocsr(), top_n=k, threshold=0.0, sort=True, n_threads=N_THREADS).tocsr()
    counts = np.diff(R.indptr)
    rows = np.repeat(np.arange(R.shape[0]), counts)
    ranks = np.arange(R.nnz) - np.repeat(R.indptr[:-1], counts) + 1
    cols = R.indices.astype(np.int64)
    keep = np.ones(R.nnz, bool) if keep_col is None else keep_col[cols]
    return rows[keep], cols[keep], R.data.astype(np.float32)[keep], ranks[keep]


def _topn(Q: sparse.csr_matrix, P: sparse.csr_matrix, k: int, keep_col: np.ndarray | None = None):
    """Top-k columns of Q @ P.T per row: (row, col, score, rank 1..k). keep_col: optional mask over P rows;
    pairs whose pool row is not kept are dropped after ranking (ranks are unchanged)."""
    if Q.shape[0] == 0 or P.shape[0] == 0:
        e = np.empty(0, dtype=np.int64)
        return e, e, np.empty(0, dtype=np.float32), e
    k = min(k, P.shape[0])
    return (_topn_gpu if USE_GPU else _topn_cpu)(Q, P, k, keep_col)


def check_gpu(Q: sparse.csr_matrix, P: sparse.csr_matrix, k: int, n_q: int = 4000, n_p: int = 1_000_000) -> dict:
    """GPU vs CPU top-k on a slice, tie-aware. Many pool records share a name, so equal scores at the k-th
    place may be kept differently; instead of comparing (row, col) pairs, compare the score at every rank
    (rank_score_diff) and recompute each GPU pair's cosine exactly on CPU (pair_score_diff). Timing runs after
    a warm-up call, so CUDA start-up is not counted."""
    Q, P = Q[:n_q], P[:n_p]
    k = min(k, P.shape[0])
    _topn_gpu(Q[:64], P, k); torch.cuda.synchronize()  # warm-up (CUDA context, cuSPARSE handles)
    t0 = time.time(); g = _topn_gpu(Q, P, k); torch.cuda.synchronize(); t1 = time.time()
    c = _topn_cpu(Q, P, k); t2 = time.time()
    by_rank = lambda x: pl.DataFrame({"r": x[0], "rk": x[3], "v": x[2]})
    j = by_rank(c).join(by_rank(g), on=["r", "rk"], how="full", suffix="_g").with_columns(pl.col("v", "v_g").fill_null(0.0))
    exact = np.asarray(Q[g[0]].multiply(P[g[1]]).sum(axis=1)).ravel()
    gp, cp = set(zip(g[0].tolist(), g[1].tolist())), set(zip(c[0].tolist(), c[1].tolist()))
    pairs = Q.shape[0] * P.shape[0]
    return {"queries": Q.shape[0], "pool": P.shape[0], "k": k,
            "rank_score_diff": float((j["v"] - j["v_g"]).abs().max() or 0.0),
            "pair_score_diff": float(np.abs(exact - g[2]).max()) if len(exact) else 0.0,
            "pair_agreement": len(gp & cp) / max(len(cp), 1),
            "gpu_ns_per_pair": (t1 - t0) / pairs * 1e9, "cpu_ns_per_pair": (t2 - t1) / pairs * 1e9}


def use_gpu(flag: bool) -> None:
    """Choose the device for all later searches (GPU only if CUDA is available)."""
    global USE_GPU
    USE_GPU = bool(flag) and torch is not None and torch.cuda.is_available()


def state_maps(records: pl.DataFrame, min_n: int = INFER_MIN_N, min_share: float = INFER_MIN_SHARE) -> dict[str, pl.DataFrame]:
    """Learn key -> state from records that have both (no labels): for city and for dept, the most frequent state
    when it covers >= min_share of min_n+ records. Used only to place records without a detected state into a
    search bucket (a compute split), never as a filter or a feature."""
    out = {}
    for key in ("city", "dept"):
        if key not in records.columns:
            continue
        c = (records.filter(pl.col(key).is_not_null() & pl.col("state").is_not_null())
             .group_by(key, "state").len()
             .with_columns(pl.col("len").sum().over(key).alias("tot"))
             .sort(key, "len", "state", descending=[False, True, False])
             .group_by(key, maintain_order=True).first())
        out[key] = c.filter((pl.col("tot") >= min_n) & (pl.col("len") / pl.col("tot") >= min_share)).select(
            key, pl.col("state").alias(f"state_from_{key}"))
    return out


def fill_states(df: pl.DataFrame, maps: dict[str, pl.DataFrame]) -> pl.DataFrame:
    """state_src = detected / city / dept / none; state = detected, else inferred from city, else from dept."""
    x = df
    for key, m in maps.items():
        x = x.join(m, on=key, how="left")
    inferred = [pl.col(f"state_from_{k}") for k in maps]
    src = pl.when(pl.col("state").is_not_null()).then(pl.lit("detected"))
    for k in maps:
        src = src.when(pl.col(f"state_from_{k}").is_not_null()).then(pl.lit(k))
    return x.with_columns(src.otherwise(pl.lit("none")).alias("state_src"),
                          pl.coalesce(pl.col("state"), *inferred).alias("state")).drop([f"state_from_{k}" for k in maps])


def buckets(s1: pl.DataFrame, pool: pl.DataFrame, scope: str) -> tuple[np.ndarray, np.ndarray]:
    if scope == "country":
        return np.full(s1.height, ALL, dtype=object), np.full(pool.height, ALL, dtype=object)
    return s1["state"].fill_null(NONE).to_numpy(), pool["state"].fill_null(NONE).to_numpy()


def forward(Q, q_bucket: np.ndarray, P, p_bucket: np.ndarray, k: int) -> pl.DataFrame:
    """For each query row: top-k pool rows among its state bucket + pool rows without a state."""
    out = []
    p_none = np.flatnonzero(p_bucket == NONE)
    for b in np.unique(q_bucket):
        qi = np.flatnonzero(q_bucket == b)
        pi = np.arange(P.shape[0]) if b == NONE else np.concatenate([np.flatnonzero(p_bucket == b), p_none])
        r, c, s, rk = _topn(Q[qi], P[pi], k)
        out.append(pl.DataFrame({"qi": qi[r], "pi": pi[c], "score": s, "rank": rk}))
    return pl.concat(out) if out else pl.DataFrame(schema={"qi": pl.Int64, "pi": pl.Int64, "score": pl.Float32, "rank": pl.Int64})


def reverse(S, s_bucket: np.ndarray, P, p_bucket: np.ndarray, m: int, keep_s: np.ndarray | None = None,
            none_fallback: bool = True) -> pl.DataFrame:
    """For each pool row: its top-m S1 rows in the same state (pool rows without a state: over all S1 of the
    country if none_fallback, else skipped). Returns (qi = S1 row, pi, score, rank,
    best), where best is the pool row's top-1 score over ALL S1 (for margin rules). keep_s: optional mask over S1
    rows; only pairs with a kept S1 are returned (all S1 still compete)."""
    out = []
    for b in np.unique(p_bucket):
        if b == NONE and not none_fallback:
            continue
        pi = np.flatnonzero(p_bucket == b)
        si = np.arange(S.shape[0]) if b == NONE else np.flatnonzero(s_bucket == b)  # no state: whole country
        r, c, s, rk = _topn(P[pi], S[si], m)
        best = np.zeros(len(pi), dtype=np.float32)
        best[r[rk == 1]] = s[rk == 1]
        f = slice(None) if keep_s is None else keep_s[si[c]]
        out.append(pl.DataFrame({"qi": si[c][f], "pi": pi[r][f], "score": s[f], "rank": rk[f], "best": best[r][f]}))
    return pl.concat(out) if out else pl.DataFrame(schema={"qi": pl.Int64, "pi": pl.Int64, "score": pl.Float32,
                                                           "rank": pl.Int64, "best": pl.Float32})


def rare_tokens(q_names: pl.Series, p_names: pl.Series, df_counts: pl.DataFrame, max_df: int = RARE_DF) -> pl.DataFrame:
    """Pool rows sharing a rare core_name token with the query (token df <= max_df in the country, so at most
    ~max_df x tokens rows per query). score = number of shared rare tokens; rank by score, ties by pool row
    (deterministic, so pieces merge exactly)."""
    rare = df_counts.filter((pl.col("df") <= max_df) & (pl.col("t").str.len_chars() >= 3)).select("t")
    tok = lambda s, name: (pl.DataFrame({name: np.arange(s.len()), "t": s.str.split(" ")})
                           .explode("t", empty_as_null=True).filter(pl.col("t").is_not_null()).unique()
                           .join(rare, on="t"))
    qt, pt = tok(q_names, "qi"), tok(p_names, "pi")
    hits = qt.join(pt, on="t").group_by("qi", "pi").agg(pl.len().cast(pl.Float32).alias("score"))
    return (hits.sort("qi", "score", "pi", descending=[False, True, False])
            .with_columns(pl.int_range(1, pl.len() + 1, dtype=pl.Int64).over("qi").alias("rank"))
            .select("qi", "pi", "score", "rank"))


def token_df(names: pl.Series) -> pl.DataFrame:
    """Document frequency of core_name tokens (distinct per record)."""
    return (pl.DataFrame({"i": np.arange(names.len()), "t": names.str.split(" ")})
            .explode("t", empty_as_null=True).filter(pl.col("t").is_not_null() & (pl.col("t") != "")).unique()
            .group_by("t").agg(pl.len().alias("df")))


def block_country(s1: pl.DataFrame, pool: pl.DataFrame, query_mask: np.ndarray, fit_corpus: dict[str, pl.Series],
                  df_counts: pl.DataFrame, k: int, m: int, scopes: tuple[str, ...] = ("state",),
                  skip: Callable[[str, str, float], bool] = lambda scope, method, pairs: False,
                  log: Callable[[str], None] = print, gpu_check: bool = False,
                  timing: list[dict] | None = None, pool_mask: np.ndarray | None = None,
                  reverse_keep: np.ndarray | None = None) -> tuple[pl.DataFrame, list[dict]]:
    """All methods for one country, for each search scope (same TF-IDF matrices). s1: ALL S1 of the split
    (reverse competition); query_mask: S1 rows to generate candidates for. skip(scope, method, pairs) may
    drop a search (e.g. over a time budget). Returns a long table (scope, qi = S1 row, pi = pool row,
    method, score, rank) and one timing row per (scope, method), appended to `timing` if given (so skip()
    can read the rates measured so far). pool_mask: pool rows that run the reverse search (a shard of the pool;
    default all). Forward and rare searches run for the query_mask rows only, so a shard = (S1 slice, pool slice)
    and the union of all shards equals one full run. reverse_keep: S1 rows whose reverse hits are returned
    (default query_mask; with pool shards pass ALL wanted S1, since any S1 can be hit from any pool shard)."""
    ts, tp = texts(s1), texts(pool)
    qrows = np.flatnonzero(query_mask)
    bk = {sc: buckets(s1, pool, sc) for sc in scopes}
    parts, timing = [], ([] if timing is None else timing)
    for rep in FORWARD:
        t0 = time.time()
        v = fit_vectorizer(fit_corpus[rep])
        S, P = v.transform(ts[rep].to_list()), v.transform(tp[rep].to_list())
        t1 = time.time()
        log(f"  {rep}: fit+transform {t1 - t0:.0f}s (vocab {len(v.vocabulary_):,}, pool nnz/row {P.nnz / max(P.shape[0], 1):.1f})")
        timing.append({"scope": "all", "method": f"{rep}_fit_transform", "search_s": t1 - t0, "queries": len(qrows),
                       "pool": P.shape[0], "pairs_evaluated": None})
        if gpu_check and USE_GPU:
            chk = check_gpu(S[qrows], P, k)
            log(f"  GPU check: {chk}")
            if chk["rank_score_diff"] > 1e-4 or chk["pair_score_diff"] > 1e-4:
                raise RuntimeError(f"GPU top-k disagrees with CPU: {chk}")
            use_gpu(chk["gpu_ns_per_pair"] < chk["cpu_ns_per_pair"])  # the faster device does all searches
            log(f"  search device from now on: {'GPU' if USE_GPU else 'CPU (faster than GPU on this data)'}")
            timing.append({"scope": "all", "method": "device_check", **chk, "device": "GPU" if USE_GPU else "CPU"})
            gpu_check = False
        for sc in scopes:
            s_bucket, p_bucket = bk[sc]
            pairs = _pairs_evaluated(s_bucket[qrows], p_bucket)
            if skip(sc, rep, pairs):
                log(f"  [{sc}] {rep}: skipped ({pairs:.2e} pairs over budget)")
                timing.append({"scope": sc, "method": rep, "pairs_evaluated": pairs, "skipped": True})
            else:
                t2 = time.time()
                f = forward(S[qrows], s_bucket[qrows], P, p_bucket, k)
                f = f.with_columns(pl.Series("qi", qrows[f["qi"].to_numpy()]))  # query-local row -> S1 row
                parts.append(f.with_columns(pl.lit(sc).alias("scope"), pl.lit(rep).alias("method")))
                timing.append({"scope": sc, "method": rep, "search_s": time.time() - t2, "queries": len(qrows),
                               "pool": P.shape[0], "pairs_evaluated": pairs})
                log(f"  [{sc}] {rep}: forward {time.time() - t2:.0f}s, {pairs:.2e} pairs, {f.height:,} rows")
            if rep == REVERSE_REP:
                prow = np.arange(P.shape[0]) if pool_mask is None else np.flatnonzero(pool_mask)
                pairs = _pairs_evaluated(p_bucket[prow], s_bucket, with_none=False)  # NONE rows: all S1
                if skip(sc, "reverse", pairs):
                    log(f"  [{sc}] reverse: skipped ({pairs:.2e} pairs over budget)")
                    timing.append({"scope": sc, "method": "reverse", "pairs_evaluated": pairs, "skipped": True})
                    continue
                t3 = time.time()
                r = reverse(S, s_bucket, P[prow], p_bucket[prow], m, keep_s=query_mask if reverse_keep is None else reverse_keep)
                r = r.with_columns(pl.Series("pi", prow[r["pi"].to_numpy()], dtype=pl.Int64))  # shard row -> pool row
                parts.append(r.with_columns(pl.lit(sc).alias("scope"), pl.lit("reverse").alias("method")))  # has "best"
                timing.append({"scope": sc, "method": "reverse", "search_s": time.time() - t3,
                               "queries": len(prow), "pool": S.shape[0], "pairs_evaluated": pairs})
                log(f"  [{sc}] reverse: {time.time() - t3:.0f}s, {pairs:.2e} pairs, {r.height:,} rows")
        del S, P
    t5 = time.time()
    rr = rare_tokens(s1["core_name"][qrows], pool["core_name"], df_counts)
    rr = rr.with_columns(pl.Series("qi", qrows[rr["qi"].to_numpy()], dtype=pl.Int64), pl.lit("rare").alias("method"))
    parts += [rr.with_columns(pl.lit(sc).alias("scope")) for sc in scopes]  # scope-independent
    timing.append({"scope": "all", "method": "rare", "search_s": time.time() - t5, "queries": len(qrows),
                   "pool": pool.height, "pairs_evaluated": None})
    log(f"  rare: {time.time() - t5:.0f}s")
    cols = ["scope", "qi", "pi", "method", "score", "rank", "best"]
    parts = [p if "best" in p.columns else p.with_columns(pl.lit(None, dtype=pl.Float32).alias("best")) for p in parts]
    return pl.concat([p.select(cols) for p in parts]), timing


def _pairs_evaluated(q_bucket: np.ndarray, p_bucket: np.ndarray, with_none: bool = True) -> float:
    """Number of (query, pool) similarity evaluations implied by the bucket sizes (cost model)."""
    pb = pl.Series(p_bucket).value_counts()
    sizes = dict(zip(pb[pb.columns[0]].to_list(), pb["count"].to_list()))
    n_none = sizes.get(NONE, 0) if with_none else 0
    total = 0.0
    for b, n in zip(*np.unique(q_bucket, return_counts=True)):
        total += n * (len(p_bucket) if b == NONE else sizes.get(b, 0) + n_none)
    return total


In [ ]:
%%writefile /tmp/src/candidates.py
"""Candidate selection on top of blocking: turn the union of search results into a SMALL list per S1.

The candidate set size is part of the ranking (organiser update), so a list should be close to the S1's true
match count (~3.4 on average). Three stages, all tuned on validation:
1. Selection rules on the blocking table (one row per S1 x candidate, rank_*/score_* per method):
   - reverse: each S2/S3 record keeps its top-1 S1, and its top-2..m S1 only if within `rev_margin` of its best;
   - forward: rank <= fwd_k and score within `fwd_margin` of the S1's best for that method (or >= `fwd_min`);
   - rare-token hits (rank <= rare_k);
   - a cap on the list length per S1, ordered by the best cosine over methods.
2. Optional pruner: a small gradient-boosted model on cheap features (blocking scores, ranks, margins,
   agreement of methods, rapidfuzz on core name and address, same city/state). It is trained on candidates of
   TRAIN S1 only and keeps a candidate if p >= tau, up to a cap. Its output is the submitted candidate set.
3. List statistics: recall, candidates per S1 (average, median, p95, S1 with no candidates included) and
   reduction ratio against all S1 x pool pairs of the country.
"""
from __future__ import annotations

import numpy as np
import polars as pl
from rapidfuzz import fuzz, process
from sklearn.ensemble import HistGradientBoostingClassifier

SEED = 42
FORWARD = ("name", "name_city", "name_addr")
METHODS = (*FORWARD, "reverse", "rare")
COSINE = (*FORWARD, "reverse")
RANK_NA = 10**6


def add_best(wide: pl.DataFrame) -> pl.DataFrame:
    """best_<method>: the S1's top forward score per method (margin reference). Needs rank/score columns."""
    return wide.with_columns(*[pl.col(f"score_{m}").max().over("scope", "s1").alias(f"best_{m}") for m in FORWARD],
                             pl.max_horizontal(*[pl.col(f"score_{m}") for m in COSINE]).fill_null(0.0).alias("order"))


def rule(rev_m: int = 0, rev_margin: float | None = None, fwd_k: int = 0, fwd_margin: float | None = None,
         fwd_min: float | None = None, rare_k: int = 0) -> pl.Expr:
    """Boolean expression over the blocking table (after add_best) for one selection config."""
    e = pl.lit(False)
    if rev_m:
        near = pl.lit(True) if rev_margin is None else pl.col("score_reverse") >= pl.col("best_reverse") - rev_margin
        e = e | ((pl.col("rank_reverse") == 1) | ((pl.col("rank_reverse") <= rev_m) & near)).fill_null(False)
    if fwd_k:
        for m in FORWARD:
            ok = pl.col(f"rank_{m}") <= fwd_k
            if fwd_margin is not None:
                near = pl.col(f"score_{m}") >= pl.col(f"best_{m}") - fwd_margin
                if fwd_min is not None:
                    near = near | (pl.col(f"score_{m}") >= fwd_min)
                ok = ok & near
            e = e | ok.fill_null(False)
    if rare_k:
        e = e | (pl.col("rank_rare") <= rare_k).fill_null(False)
    return e


def select(wide: pl.DataFrame, expr: pl.Expr, cap: int | None = None, order: str = "order") -> pl.DataFrame:
    """Rows passing expr, then at most `cap` per (scope, S1) by descending `order`."""
    out = wide.filter(expr)
    if cap:
        out = out.filter(pl.col(order).rank("ordinal", descending=True).over("scope", "s1") <= cap)
    return out


# ---------------------------------------------------------------- pruner

def features(cand: pl.DataFrame, s1_attr: pl.DataFrame, pool_attr: pl.DataFrame) -> pl.DataFrame:
    """Cheap per-pair features. cand: blocking rows (after add_best); *_attr: id + core_name, addr_norm, city, state."""
    a = s1_attr.select(pl.col("entity_id").alias("s1"), *[pl.col(c).alias(f"q_{c}") for c in ("core_name", "addr_norm", "city", "state")])
    b = pool_attr.select(pl.col("entity_id").alias("cand"), *[pl.col(c).alias(f"p_{c}") for c in ("core_name", "addr_norm", "city", "state")])
    x = cand.join(a, on="s1", how="left").join(b, on="cand", how="left")
    fz = lambda scorer, c: process.cpdist(x[f"q_{c}"].fill_null("").to_list(), x[f"p_{c}"].fill_null("").to_list(),
                                          scorer=scorer, workers=-1).astype(np.float32)
    x = x.with_columns(
        pl.Series("f_name_tsr", fz(fuzz.token_set_ratio, "core_name")),
        pl.Series("f_name_ratio", fz(fuzz.ratio, "core_name")),
        pl.Series("f_addr_tsr", fz(fuzz.token_set_ratio, "addr_norm")),
        (pl.col("q_city") == pl.col("p_city")).cast(pl.Int8).fill_null(-1).alias("f_same_city"),
        (pl.col("q_state") == pl.col("p_state")).cast(pl.Int8).fill_null(-1).alias("f_same_state"),
    )
    return x


def feature_cols() -> list[str]:
    return [*[f"score_{m}" for m in METHODS], *[f"rank_{m}" for m in METHODS], *[f"gap_{m}" for m in FORWARD],
            "gap_reverse", "n_methods", "list_rank", "list_size",
            "f_name_tsr", "f_name_ratio", "f_addr_tsr", "f_same_city", "f_same_state"]


def matrix(x: pl.DataFrame) -> np.ndarray:
    """Feature matrix; x must carry list_rank / list_size (add_list_features on the whole list)."""
    x = x.with_columns(
        *[(pl.col(f"best_{m}") - pl.col(f"score_{m}")).alias(f"gap_{m}") for m in FORWARD],
        (pl.col("best_reverse") - pl.col("score_reverse")).alias("gap_reverse"),
        pl.sum_horizontal(*[pl.col(f"rank_{m}").is_not_null() for m in METHODS]).alias("n_methods"),
    ).with_columns(*[pl.col(f"rank_{m}").fill_null(RANK_NA) for m in METHODS])
    return x.select(feature_cols()).to_numpy().astype(np.float32)  # NaN (missing score) is handled by the model


def train_pruner(x: pl.DataFrame, y: np.ndarray) -> HistGradientBoostingClassifier:
    m = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, max_leaf_nodes=31, min_samples_leaf=50,
                                       l2_regularization=1.0, random_state=SEED)
    return m.fit(matrix(x), y)


def add_list_features(cand: pl.DataFrame) -> pl.DataFrame:
    """Per-S1 list features, computed on the WHOLE starting list (before any sampling or chunking)."""
    return cand.with_columns(pl.col("order").rank("ordinal", descending=True).over("scope", "s1").alias("list_rank"),
                             pl.len().over("scope", "s1").alias("list_size"))


def score(model, cand: pl.DataFrame, s1_attr: pl.DataFrame, pool_attr: pl.DataFrame, chunk: int = 2_000_000) -> np.ndarray:
    """Pruner probability per row of cand (after add_list_features), in row chunks to bound memory."""
    return np.concatenate([model.predict_proba(matrix(features(cand[lo:lo + chunk], s1_attr, pool_attr)))[:, 1]
                           for lo in range(0, cand.height, chunk)] or [np.empty(0)])


def training_rows(cand: pl.DataFrame, max_rows: int, seed: int = SEED) -> pl.DataFrame:
    """All positives plus a seeded sample of negatives, at most max_rows (after add_list_features)."""
    if cand.height <= max_rows:
        return cand
    pos, neg = cand.filter(pl.col("is_match")), cand.filter(~pl.col("is_match"))
    return pl.concat([pos, neg.sample(max(max_rows - pos.height, 0), seed=seed)])


def prune(x: pl.DataFrame, p: np.ndarray, tau: float, cap: int | None = None) -> pl.DataFrame:
    """Keep candidates with pruner probability >= tau, at most `cap` per (scope, S1) by descending probability."""
    out = x.with_columns(pl.Series("p_keep", p)).filter(pl.col("p_keep") >= tau)
    if cap:
        out = out.filter(pl.col("p_keep").rank("ordinal", descending=True).over("scope", "s1") <= cap)
    return out


# ---------------------------------------------------------------- list statistics

def list_stats(found: pl.DataFrame, truth: pl.DataFrame, s1s: pl.DataFrame, pool_size: dict[str, int]) -> pl.DataFrame:
    """found: (s1, cand) selected; truth: (s1, cand) true pairs; s1s: (s1, country) of all evaluated S1;
    pool_size: S2+S3 records per country. Per country + ALL: recall, avg/median/p95 candidates per S1 (S1 with
    no candidates count as 0), share of S1 with an empty list, reduction ratio vs all S1 x pool pairs."""
    per_s1 = lambda df, name: df.join(s1s.select("s1"), on="s1", how="semi").group_by("s1").len(name)
    d = (s1s.join(per_s1(found, "n"), on="s1", how="left")
         .join(per_s1(truth, "true"), on="s1", how="left")
         .join(per_s1(truth.join(found.select("s1", "cand"), on=["s1", "cand"], how="semi"), "hit"), on="s1", how="left")
         .with_columns(pl.col("n", "true", "hit").fill_null(0),
                       pl.col("country").replace_strict(pool_size, default=0, return_dtype=pl.Float64).alias("pool")))
    d = pl.concat([d, d.with_columns(pl.lit("ALL").alias("country"))])
    return d.group_by("country").agg(
        (pl.col("hit").sum() / pl.col("true").sum()).alias("recall"),
        pl.col("n").mean().alias("avg"), pl.col("n").median().alias("median"),
        pl.col("n").quantile(0.95, "nearest").alias("p95"), (pl.col("n") == 0).mean().alias("empty_share"),
        (1 - pl.col("n").sum() / pl.col("pool").sum()).alias("reduction_ratio"),
        pl.len().alias("n_s1"), pl.col("true").sum(), pl.col("n").sum().alias("cands"),
    ).sort(pl.col("country") == "ALL", "country")


def pareto(points: pl.DataFrame, x: str = "avg", y: str = "recall") -> pl.DataFrame:
    """Configs not dominated on (fewer candidates, higher recall)."""
    p = points.sort(x, -pl.col(y))
    best, keep = -1.0, []
    for v in p[y].to_list():
        keep.append(v > best)
        best = max(best, v)
    return p.filter(pl.Series(keep))


In [ ]:
%%writefile /tmp/src/run_blocking_eval.py
"""Blocking + candidate selection evaluation on the validation split against the FULL train S2/S3 pool.

The candidate set size is ranked alongside F0.5, so every design is judged on two axes: recall of true pairs
AND candidates per S1 (average, median, p95). Configs evaluated per scope:
  fixed     forward top-k (all methods) + reverse top-m (no margins), + rare tokens
  reverse   reverse top-1/2/3 with a margin rule, alone or with a small forward top-k
  adaptive  forward within a margin of the S1's best (or above a minimum cosine), up to k per method, + reverse
  pruned    a cheap pruner (candidates.py) on a generous union, keep p >= tau up to a cap; trained on candidates
            of --train-queries TRAIN-split S1 (never validation S1)
Operating points: the best recall at <= 5, <= 10 and <= 20 candidates per S1 on average.

Usage: python src/run_blocking_eval.py --norm-dir artifacts/normalised --data-dir data_parquet \
           --split outputs/eda/g25_split.parquet --out-dir artifacts/blocking_eval
--norm-dir / --split may be folders to search (e.g. /kaggle/input). Outputs in --out-dir:
  curve.csv (every config: family, scope, config, per-country and ALL recall / avg / median / p95 / reduction),
  pareto.csv, operating_points.csv, recall_groups.csv, timing.csv, test_estimate.csv, blocking_curve.png,
  val_candidates.parquet (validation S1 x candidates of the generous union, with pruner probability),
  blocking_report.md, config.json.
Pieces: `--stage search --shard i/n [--countries X]` writes one piece of the search (pieces/piece_*.parquet);
`--stage evaluate --pieces DIR...` merges complete piece sets and evaluates; `--stage all` does both.
Scopes: "state" (searches within state buckets) or "country" (whole country). Country-scope searches whose
projected time would pass --budget-min are skipped and reported.
"""
import argparse
import json
import pickle
import re
import resource
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl
from rapidfuzz import fuzz, process

try:
    sys.path.insert(0, str(Path(__file__).resolve().parent))
except NameError:  # Kaggle notebook: modules are written to /tmp/src
    sys.path.insert(0, "/tmp/src")
import blocking  # noqa: E402
from blocking import (FORWARD, NONE, SCOPES, _pairs_evaluated, block_country, buckets, fill_states, gpu_info,  # noqa: E402
                      state_maps, texts, token_df)
from candidates import (METHODS, add_best, add_list_features, feature_cols, features, list_stats, pareto,  # noqa: E402
                        prune, rule, score, select, train_pruner, training_rows)

SEED = 42
TARGETS = [5, 7, 10, 20]  # operating points: best recall at <= this many candidates per S1 on average (7 = default)
DEFAULT_TARGET = 7
COLS = ["entity_id", "country", "business_name", "core_name", "city", "addr_norm", "state", "dept"]
NON_LATIN = r"[\p{L}&&[^\p{Latin}]]"
T0 = time.time()
UNIONS = {"u20": dict(rev_m=3, fwd_k=20, rare_k=10), "u50": dict(rev_m=5, fwd_k=50, rare_k=20)}  # pruner starting lists
PRUNER_TRAIN_MAX = 6_000_000  # cap on pruner training rows (all positives kept, negatives sampled)
FEATURE_CHUNK = 2_000_000  # rows per feature/predict chunk (bounded memory)


def configs() -> list[tuple[str, dict, int | None]]:
    """(family, rule kwargs, cap) for every selection config."""
    out = []
    for k in [0, 1, 2, 3, 5, 10, 20, 50]:
        for m in [0, 1, 2, 3, 5]:
            for rare in [0, 10]:
                if k or m or rare:
                    out.append(("fixed", dict(fwd_k=k, rev_m=m, rare_k=rare), None))
    for m in [1, 2, 3]:
        for mg in [0.02, 0.05, 0.1, 0.2, None]:
            if m == 1 and mg is not None:
                continue
            for k in [0, 1, 2]:
                out.append(("reverse", dict(rev_m=m, rev_margin=mg, fwd_k=k), None))
    for k in [3, 5, 10, 20]:
        for mg in [0.02, 0.05, 0.1, 0.2]:
            for mn in [None, 0.9]:
                for rv in [(1, None), (2, 0.05), (2, 0.1), (3, 0.1)]:
                    for cap in [None, 10, 20]:
                        out.append(("adaptive", dict(fwd_k=k, fwd_margin=mg, fwd_min=mn, rev_m=rv[0], rev_margin=rv[1]), cap))
    return out


def log(msg: str) -> None:
    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20
    print(f"[{time.time() - T0:7.1f}s peak {peak:5.1f} GB] {msg}", flush=True)


def find(root: Path, name: str, parent: str | None = None) -> Path:
    if root.is_file():
        return root
    hits = sorted(p for p in root.rglob(name) if parent is None or p.parent.name == parent)
    if not hits:
        raise FileNotFoundError(f"{name} (parent {parent}) not found under {root}")
    return hits[0]


def md(df: pl.DataFrame, max_rows: int = 60) -> str:
    df = df.head(max_rows)
    fmt = lambda v: f"{v:.4f}" if isinstance(v, float) else str(v)
    rows = [" | ".join(df.columns), " | ".join("---" for _ in df.columns)] + [" | ".join(fmt(v) for v in r) for r in df.iter_rows()]
    return "\n".join(f"| {r} |" for r in rows)


def test_pairs(te: dict, country: str, scopes: list[str]) -> list[dict]:
    """Similarity evaluations a test run would need for this country, per scope (same cost model as timing)."""
    t1 = te[1].filter(pl.col("country") == country)
    tp = pl.concat([te[2].filter(pl.col("country") == country), te[3].filter(pl.col("country") == country)])
    out = []
    for sc in scopes:
        tsb, tpb = buckets(t1, tp, sc)
        out.append({"scope": sc, "country": country, "test_s1": t1.height, "test_pool": tp.height,
                    "forward_pairs": _pairs_evaluated(tsb, tpb),
                    "reverse_pairs": _pairs_evaluated(tpb[tpb != NONE], tsb, with_none=False)})
    return out


rates: dict[str, float] = {}  # seconds per similarity evaluation, "forward" / "reverse", measured so far
BUDGET_S = 600 * 60.0


def measured_rates(timing: list[dict]) -> dict[str, float]:
    out = {}
    for kind in ("forward", "reverse"):
        rows = [t for t in timing if t.get("pairs_evaluated") and not t.get("skipped")
                and (t["method"] == "reverse") == (kind == "reverse")]
        if rows:
            out[kind] = sum(t["search_s"] for t in rows) / sum(t["pairs_evaluated"] for t in rows)
    return out


def over_budget(tim: list[dict]):
    """skip() for block_country: skip a search if its projected time (pairs x measured rate) would pass the
    budget, keeping 20% of the budget for evaluation. State-scope searches are never skipped."""
    def skip(scope: str, method: str, pairs: float) -> bool:
        if scope == "state":
            return False
        r = {**rates, **measured_rates(tim)}.get("reverse" if method == "reverse" else "forward")
        return r is not None and (time.time() - T0) + pairs * r > 0.8 * BUDGET_S
    return skip


def plot(curve: pl.DataFrame, front: pl.DataFrame, ops: pl.DataFrame, floor: float, path: Path) -> None:
    """Recall vs average candidates per S1 (log x), one panel per scope; Pareto front and operating points."""
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        log("matplotlib not available; skipping the plot (curve.csv has the data)")
        return
    colors = {"fixed": "#2a78d6", "reverse": "#eb6834", "adaptive": "#1baf7a", "pruned": "#eda100"}  # fixed order
    scopes = curve["scope"].unique(maintain_order=True).to_list()
    fig, axes = plt.subplots(1, len(scopes), figsize=(6.5 * len(scopes), 4.8), squeeze=False, facecolor="#fcfcfb")
    for ax, sc in zip(axes[0], scopes):
        ax.set_facecolor("#fcfcfb")
        c = curve.filter((pl.col("scope") == sc) & (pl.col("country") == "ALL"))
        for fam, col in colors.items():
            d = c.filter(pl.col("family") == fam)
            if d.height:
                ax.scatter(d["avg"], d["recall"], s=16, color=col, alpha=0.7, linewidths=0, label=fam)
        f = front.filter(pl.col("scope") == sc).sort("avg")
        ax.plot(f["avg"], f["recall"], color="#3d3d3a", lw=1.5, label="Pareto front")
        ax.axvline(floor, color="#8a8a85", lw=1, ls="--")
        ax.text(floor, 0.02 + ax.get_ylim()[0], f" true matches per S1 = {floor:.2f}", color="#5f5e5a", fontsize=8, va="bottom")
        for r in ops.filter((pl.col("scope") == sc) & (pl.col("country") == "ALL")).iter_rows(named=True):
            ax.scatter([r["avg"]], [r["recall"]], s=64, facecolors="none", edgecolors="#1a1a19", linewidths=1.5, zorder=5)
            ax.annotate(f"<= {r['target']}: {r['recall']:.3f} @ {r['avg']:.1f}", (r["avg"], r["recall"]),
                        textcoords="offset points", xytext=(6, -12), fontsize=8, color="#1a1a19")
        ax.set_xscale("log")
        ax.set_xlabel("average candidates per validation S1 (log scale)", color="#3d3d3a")
        ax.set_ylabel("recall of true pairs", color="#3d3d3a")
        ax.set_title(f"Blocking: recall vs candidate set size ({sc} scope)", color="#1a1a19", fontsize=11, loc="left")
        ax.grid(True, color="#e8e7e1", lw=0.8)
        for s in ax.spines.values():
            s.set_color("#c3c2b7")
        ax.tick_params(colors="#5f5e5a")
        ax.legend(frameon=False, fontsize=8, loc="lower right")
    fig.tight_layout()
    fig.savefig(path, dpi=130, facecolor="#fcfcfb")
    plt.close(fig)


def bucket_stats(s1: pl.DataFrame, pool: pl.DataFrame) -> list[dict]:
    """State-bucket facts for one country: where states come from, and the largest within-state search groups
    (pairs = S1 in the bucket x (pool in the bucket + pool without a state); S1 without a state search everything)."""
    rows = [{"method": "state_src", "side_table": name, "src": src, "n": n}
            for name, df in (("s1", s1), ("pool", pool)) for src, n in df["state_src"].value_counts().iter_rows()]
    sb, pb = s1["state"].fill_null(NONE), pool["state"].fill_null(NONE)
    ps = dict(pb.value_counts().iter_rows())
    n_none = ps.get(NONE, 0)
    grp = [{"bucket": b, "s1": n, "pool": pool.height if b == NONE else ps.get(b, 0) + n_none} for b, n in sb.value_counts().iter_rows()]
    tot = sum(g["s1"] * g["pool"] for g in grp) or 1
    for g in sorted(grp, key=lambda g: -g["s1"] * g["pool"])[:5]:
        rows.append({"method": "bucket", **g, "pairs_share": g["s1"] * g["pool"] / tot})
    rows.append({"method": "bucket_summary", "n_buckets": len(grp), "pairs": tot, "s1_total": s1.height, "pool_total": pool.height})
    return rows


def to_wide(long: pl.DataFrame) -> pl.DataFrame:
    """Search results (one row per method hit) -> one row per (scope, S1, candidate) with rank/score per method,
    the pool record's best reverse score and the S1's best forward score per method (add_best)."""
    best_rev = (long.filter(pl.col("method") == "reverse").group_by("scope", "s1", "cand")
                .agg(pl.col("best").max().alias("best_reverse")))
    wide = long.pivot(on="method", index=["scope", "s1", "cand"], values=["rank", "score"], aggregate_function="min")
    for mth in METHODS:
        for c in ("rank", "score"):
            if f"{c}_{mth}" not in wide.columns:
                wide = wide.with_columns(pl.lit(None, dtype=pl.Int64 if c == "rank" else pl.Float32).alias(f"{c}_{mth}"))
    return add_best(wide.join(best_rev, on=["scope", "s1", "cand"], how="left")).sort("scope", "s1", "cand")


def filled_states(tr: dict, te: dict, countries: list[str]) -> dict[int, pl.DataFrame]:
    """tr with missing states inferred per country (same maps as search_piece: learned on train + test records)."""
    out = {k: [] for k in tr}
    for country in countries:
        cc = lambda d: d.filter(pl.col("country") == country)
        maps = state_maps(pl.concat([cc(t) for t in [*tr.values(), *te.values()]], how="diagonal_relaxed"))
        for k in tr:
            out[k].append(fill_states(cc(tr[k]), maps))
    return {k: pl.concat(v) for k, v in out.items()}


def search_piece(a, tr: dict, te: dict, role: pl.DataFrame, country: str, shard: int, n_shards: int, out: Path) -> None:
    """One piece of the search: S1 rows with row % n == shard (forward, rare) and pool rows with row % n == shard
    (reverse). The TF-IDF vocabulary is rebuilt identically in every piece (fixed seed, same corpus), so the union
    of all n pieces equals one full run. --side train queries the role S1 (validation + pruner train sample) against
    the train pool; --side test queries a --test-fraction sample of test S1 against the full test pool (no labels).
    Missing states are inferred from city / dept before bucketing (state_maps, learned on all records of the
    country, no labels). Writes piece_<side>_<country>_<i>of<n>.parquet and its timing JSON."""
    tag = f"piece_{a.side}_{country}_{shard}of{n_shards}"
    log(f"{tag}: start")
    src = tr if a.side == "train" else te
    cc = lambda d: d.filter(pl.col("country") == country)
    train_all = pl.concat([cc(t) for t in tr.values()], how="diagonal_relaxed")
    test_all = pl.concat([cc(t) for t in te.values()], how="diagonal_relaxed")
    maps = state_maps(pl.concat([train_all, test_all], how="diagonal_relaxed"))
    s1 = fill_states(cc(src[1]), maps)
    pool = fill_states(pl.concat([cc(src[2]), cc(src[3])]), maps)
    corpus = {rep: pl.concat([texts(train_all)[rep], texts(test_all)[rep]]) for rep in FORWARD}
    df_counts = token_df(pl.concat([train_all["core_name"], test_all["core_name"]]))
    in_shard = lambda n: np.arange(n) % n_shards == shard
    if a.side == "train":
        wanted = s1["entity_id"].is_in(role["s1"].implode()).to_numpy()
    else:
        frac = dict(x.split("=") for x in a.test_fraction.split(",")).get(country, "1")
        wanted = np.random.default_rng(SEED).random(s1.height) < float(frac)
    qmask = wanted & in_shard(s1.height)
    tim: list[dict] = []
    long, _ = block_country(s1, pool, qmask, corpus, df_counts, a.k_max, a.m_max, scopes=tuple(a.scopes),
                            skip=over_budget(tim), log=log, gpu_check=shard == 0, timing=tim,
                            pool_mask=in_shard(pool.height), reverse_keep=wanted)
    rates.update(measured_rates(tim))
    long = long.with_columns(pl.Series("s1", s1["entity_id"].to_numpy()[long["qi"].to_numpy()]),
                             pl.Series("cand", pool["entity_id"].to_numpy()[long["pi"].to_numpy()])).drop("qi", "pi")
    long.write_parquet(out / f"{tag}.parquet")
    tim += bucket_stats(s1, pool) + [{"method": "queried", "s1": int(wanted.sum()), "s1_total": s1.height}]
    (out / f"{tag}_timing.json").write_text(json.dumps([{"country": country, "side": a.side, "shard": f"{shard}/{n_shards}", **t}
                                                        for t in tim], default=float))
    log(f"{tag}: {long.height:,} rows written")


def load_pieces(dirs: list[Path], countries: list[str], side: str = "train") -> tuple[pl.DataFrame, list[dict]]:
    """Merge piece files of one side; every country needs a complete set 0..n-1 of one n."""
    files = sorted({f for d in dirs for f in d.rglob(f"piece_{side}_*of*.parquet")})
    have: dict[str, dict[int, set[int]]] = {}
    for f in files:
        c, i, n = re.fullmatch(rf"piece_{side}_(.+)_(\d+)of(\d+)\.parquet", f.name).groups()
        have.setdefault(c, {}).setdefault(int(n), set()).add(int(i))
    for c in countries:
        ok = [n for n, got in have.get(c, {}).items() if got == set(range(n))]
        if len(ok) != 1:
            raise RuntimeError(f"pieces for {c} incomplete or mixed: {have.get(c)} (files under {dirs})")
    long = pl.concat([pl.read_parquet(f) for f in files])
    timing = [t for f in files for t in json.loads(f.with_name(f.stem + "_timing.json").read_text())]
    log(f"merged {len(files)} pieces: {long.height:,} rows")
    return long, timing


def main() -> None:
    global BUDGET_S
    ap = argparse.ArgumentParser()
    ap.add_argument("--norm-dir", default="artifacts/normalised")
    ap.add_argument("--data-dir", default="data_parquet")
    ap.add_argument("--split", default="outputs/eda/g25_split.parquet")
    ap.add_argument("--out-dir", default="artifacts/blocking_eval")
    ap.add_argument("--k-max", type=int, default=50)
    ap.add_argument("--m-max", type=int, default=5)
    ap.add_argument("--train-queries", type=int, default=100_000, help="TRAIN-split S1 queried to fit the pruner (0 = no pruner)")
    ap.add_argument("--scopes", nargs="+", default=list(SCOPES), choices=SCOPES)
    ap.add_argument("--budget-min", type=float, default=600, help="wall-clock budget for the searches (minutes)")
    ap.add_argument("--stage", choices=["all", "search", "evaluate"], default="all",
                    help="search: write result pieces only; evaluate: merge pieces and evaluate; all: both")
    ap.add_argument("--shard", default="0/1", help="search stage: piece i/n (S1 queries and pool rows split n ways)")
    ap.add_argument("--countries", nargs="*", default=None, help="search stage: countries to search (default all)")
    ap.add_argument("--pieces", nargs="*", default=None, help="evaluate stage: folders holding piece files")
    ap.add_argument("--side", choices=["train", "test"], default="train", help="search stage: train (labelled) or test")
    ap.add_argument("--test-fraction", default="France=1,US=0.1,India=0.1",
                    help="search stage, test side: share of test S1 queried per country (pool is always full)")
    a = ap.parse_args()
    BUDGET_S = a.budget_min * 60
    log(f"search device: {gpu_info()}; scopes {a.scopes}; budget {a.budget_min:.0f} min")
    out = Path(a.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    norm = lambda sp, k, cols=COLS: pl.read_parquet(find(Path(a.norm_dir), f"{sp}_source{k}.parquet", "normalised"), columns=cols)
    split = pl.read_parquet(find(Path(a.split), "g25_split.parquet"))
    tr = {k: norm("train", k) for k in (1, 2, 3)}
    te = {k: norm("test", k) for k in (1, 2, 3)}
    present = tr[1]["entity_id"]  # the local sample holds only part of the split
    val_ids = split.filter((pl.col("role") == "val") & pl.col("source1_entity_id").is_in(present.implode()))["source1_entity_id"]
    trn = split.filter((pl.col("role") == "train") & pl.col("source1_entity_id").is_in(present.implode()))["source1_entity_id"]
    trn_ids = trn.sample(min(a.train_queries, trn.len()), seed=SEED) if a.train_queries else trn.head(0)
    role = pl.concat([pl.DataFrame({"s1": val_ids, "role": "val"}), pl.DataFrame({"s1": trn_ids, "role": "train"})])
    gt = pl.read_parquet(find(Path(a.data_dir), "train_ground_truth.parquet"))
    truth = (
        gt.filter(pl.col("source1_entity_id").is_in(role["s1"].implode()))
        .select(pl.col("source1_entity_id").alias("s1"), pl.col("matched_entity_ids").str.split(",").alias("cand"))
        .explode("cand", empty_as_null=True).filter(pl.col("cand").is_not_null() & (pl.col("cand") != ""))
    )
    log(f"loaded; {val_ids.len():,} validation S1 + {trn_ids.len():,} train S1 (pruner), {truth.height:,} true pairs")

    countries = tr[1]["country"].unique().sort().to_list()
    shard, n_shards = (int(x) for x in a.shard.split("/"))
    if a.stage in ("all", "search"):
        (out / "pieces").mkdir(exist_ok=True)
        for country in a.countries or (countries if a.side == "train" else te[1]["country"].unique().sort().to_list()):
            search_piece(a, tr, te, role, country, shard, n_shards, out / "pieces")
        if a.stage == "search":
            log("search stage done")
            return
    piece_dirs = [Path(d) for d in a.pieces] if a.pieces else [out / "pieces"]
    long, timing = load_pieces(piece_dirs, countries)

    # per-country facts for the evaluation (no search needed)
    test_est, attrs, pool_size = [], [], {}
    for country in countries:
        s1 = tr[1].filter(pl.col("country") == country)
        pool = pl.concat([tr[2].filter(pl.col("country") == country), tr[3].filter(pl.col("country") == country)])
        pool_size[country] = pool.height
        test_core = pl.concat([t.filter(pl.col("country") == country)["core_name"] for t in te.values()])
        cnt = pl.concat([s1["core_name"], pool["core_name"], test_core]).alias("core_name").value_counts(name="n_core")
        attrs.append(s1.join(role, left_on="entity_id", right_on="s1", how="semi")
                     .select(pl.col("entity_id").alias("s1"), "country", "core_name").join(cnt, on="core_name", how="left"))
    for country in sorted(set(te[1]["country"].unique().to_list()) | set(countries)):
        test_est += test_pairs(te, country, a.scopes)  # includes test-only countries (France)

    # ---- one row per (scope, S1, candidate): rank/score per method, reverse best, forward best per S1
    wide = to_wide(long)
    del long
    wide = wide.join(truth.select("s1", "cand", pl.lit(True).alias("is_match")), on=["s1", "cand"], how="left").with_columns(
        pl.col("is_match").fill_null(False)).join(role, on="s1").sort("scope", "s1", "cand")  # order-independent of pieces
    log(f"blocking table: {wide.height:,} (scope, S1, cand) rows")

    val_s1 = pl.concat(attrs).join(role.filter(pl.col("role") == "val"), on="s1", how="semi")
    s1s = val_s1.select("s1", "country")
    t = truth.join(s1s, on="s1", how="semi")
    floor = t.height / max(s1s.height, 1)
    # validation halves by id parity (ids carry no information): configs are chosen on "tune", reported on "report"
    half = pl.col("s1").str.extract(r"(\d+)$").cast(pl.Int64) % 2
    s1s_tune, s1s_rep = s1s.filter(half == 0), s1s.filter(half == 1)
    t_tune, t_rep = t.join(s1s_tune, on="s1", how="semi"), t.join(s1s_rep, on="s1", how="semi")
    wv = wide.filter(pl.col("role") == "val")
    tim_rows = []

    # ---- pruner: trained on TRAIN-split S1 candidates of each starting list (union), applied to validation S1.
    # Leakage guard: the pruner never sees a validation S1 (asserted), and tau / cap are chosen on the tune half.
    assert not set(trn_ids.to_list()) & set(val_ids.to_list()), "pruner train S1 overlap validation S1"
    # features see the same filled states as the buckets (fewer "missing" values, less shift for France)
    trf = filled_states(tr, te, countries)
    pool_attr = pl.concat([trf[2], trf[3]]).select("entity_id", "core_name", "addr_norm", "city", "state")
    s1_attr = trf[1].select("entity_id", "core_name", "addr_norm", "city", "state")
    pruned: dict[tuple[str, str], tuple[pl.DataFrame, np.ndarray]] = {}
    audit = []
    if trn_ids.len():
        for sc in a.scopes:
            for un, ukw in UNIONS.items():
                t0 = time.time()
                ws = add_list_features(wide.filter((pl.col("scope") == sc) & rule(**ukw)))
                xt = features(training_rows(ws.filter(pl.col("role") == "train"), PRUNER_TRAIN_MAX), s1_attr, pool_attr)
                t1 = time.time()
                model = train_pruner(xt, xt["is_match"].to_numpy())
                t2 = time.time()
                base = ws.filter(pl.col("role") == "val")
                pv = score(model, base, s1_attr, pool_attr, FEATURE_CHUNK)
                t3 = time.time()
                pruned[(sc, un)] = (base.select("scope", "s1", "cand", "is_match"), pv)
                with open(out / f"pruner_{sc}_{un}.pkl", "wb") as f:
                    pickle.dump({"model": model, "features": feature_cols(), "union": ukw, "scope": sc}, f)
                tim_rows.append({"scope": sc, "method": f"pruner_{un}", "train_rows": xt.height, "val_rows": base.height,
                                 "features_s": t1 - t0, "fit_s": t2 - t1, "predict_s": t3 - t2,
                                 "s_per_row": (t3 - t2) / max(base.height, 1)})
                log(f"[{sc}] pruner {un}: {xt.height:,} train rows ({xt['is_match'].mean():.3f} positive), fit {t2 - t1:.0f}s; "
                    f"{base.height:,} val rows scored in {t3 - t2:.0f}s")
                # feature audit: missing-value share of the location features, per country (train rows)
                xa = xt.join(tr[1].select(pl.col("entity_id").alias("s1"), "country"), on="s1")
                audit += xa.group_by("country").agg(
                    pl.lit(sc).alias("scope"), pl.lit(un).alias("union"), pl.len().alias("rows"),
                    (pl.col("f_same_state") == -1).mean().alias("same_state_missing"),
                    (pl.col("f_same_city") == -1).mean().alias("same_city_missing"),
                    pl.col("is_match").mean().alias("positive_share")).to_dicts()
                del xt, xa, ws

    def found_for(sc: str, family: str, config: str) -> pl.DataFrame:
        cfg = json.loads(config)
        if family == "pruned":
            base, pv = pruned[(sc, cfg["union"])]
            return prune(base, pv, cfg["tau"], cfg.get("cap"))
        cap = cfg.pop("cap", None)
        return select(wv.filter(pl.col("scope") == sc), rule(**cfg), cap)

    # ---- every config: recall and list size on the TUNE half of validation
    rows = []
    for sc in a.scopes:
        w = wv.filter(pl.col("scope") == sc)
        for fam, kw, cap in configs():
            found = select(w, rule(**kw), cap).select("s1", "cand")
            name = json.dumps({**{k: v for k, v in kw.items() if v not in (0, None)}, **({"cap": cap} if cap else {})})
            rows += [{"family": fam, "scope": sc, "config": name, **r} for r in list_stats(found, t_tune, s1s_tune, pool_size).iter_rows(named=True)]
        for (psc, un), (base, pv) in pruned.items():
            if psc != sc:
                continue
            for tau in [0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5]:
                for cap in [None, 3, 5, 7, 10, 20]:
                    found = prune(base, pv, tau, cap).select("s1", "cand")
                    name = json.dumps({"union": un, "tau": tau, **({"cap": cap} if cap else {})})
                    rows += [{"family": "pruned", "scope": sc, "config": name, **r}
                             for r in list_stats(found, t_tune, s1s_tune, pool_size).iter_rows(named=True)]
        log(f"[{sc}] configs evaluated")
    curve = pl.DataFrame(rows)
    curve.write_csv(out / "curve.csv")
    allc = curve.filter(pl.col("country") == "ALL")
    front = pl.concat([pareto(allc.filter(pl.col("scope") == sc)) for sc in a.scopes])
    front.write_csv(out / "pareto.csv")

    # ---- operating points: chosen on the TUNE half (best recall at <= target candidates per S1, ALL countries),
    # reported on the REPORT half (unbiased), per country
    ops = []
    for sc in a.scopes:
        for tg in TARGETS:
            c = allc.filter((pl.col("scope") == sc) & (pl.col("avg") <= tg)).sort("recall", "avg", descending=[True, False])
            if c.height:
                r = c.row(0, named=True)
                st = list_stats(found_for(sc, r["family"], r["config"]).select("s1", "cand"), t_rep, s1s_rep, pool_size)
                ops += [{"target": tg, "family": r["family"], "scope": sc, "config": r["config"], "tune_recall": r["recall"],
                         "tune_avg": r["avg"], **x} for x in st.iter_rows(named=True)]
    ops = pl.DataFrame(ops)
    ops.write_csv(out / "operating_points.csv")
    plot(curve, front, ops, floor, out / "blocking_curve.png")

    # ---- group recall at the operating points
    matches = pl.concat([tr[2].select("entity_id", "core_name", "business_name", "state"),
                         tr[3].select("entity_id", "core_name", "business_name", "state")]).rename(
        {"entity_id": "cand", "core_name": "m_core", "business_name": "m_name", "state": "m_state"})
    s1_state = tr[1].select(pl.col("entity_id").alias("s1"), pl.col("state").alias("s1_state"))
    g = t_rep.join(val_s1.select("s1", "country", "core_name", "n_core"), on="s1").join(matches, on="cand", how="left").join(s1_state, on="s1")
    g = g.with_columns(pl.Series("tsr", process.cpdist(g["core_name"].to_list(), g["m_core"].fill_null("").to_list(),
                                                       scorer=fuzz.token_set_ratio, workers=-1)))
    groups = {
        "all": pl.lit(True),
        "common name (core_name 5+ in country)": pl.col("n_core") >= 5,
        "low name similarity (tsr < 70)": pl.col("tsr") < 70,
        "Indian-script match record": pl.col("m_name").str.contains(NON_LATIN),
        "state differs (both known)": pl.col("m_state").is_not_null() & (pl.col("m_state") != pl.col("s1_state")),
        "match has no state": pl.col("m_state").is_null(),
    }
    rows = []
    for r in ops.filter(pl.col("country") == "ALL").iter_rows(named=True):
        sc = r["scope"]
        found = found_for(sc, r["family"], r["config"])
        gg = g.join(found.select("s1", "cand", pl.lit(True).alias("found")), on=["s1", "cand"], how="left").with_columns(
            pl.col("found").fill_null(False))
        for gname, cond in groups.items():
            sub = gg.filter(cond)
            for country, d in [("ALL", sub)] + [(c, x) for (c,), x in sub.group_by("country")]:
                rows.append({"scope": sc, "target": r["target"], "family": r["family"], "config": r["config"], "group": gname,
                             "country": country, "true_pairs": d.height, "recall": d["found"].mean() if d.height else None})
    grp = pl.DataFrame(rows).sort("scope", "target", "group", "country")
    grp.write_csv(out / "recall_groups.csv")

    # ---- validation candidates of the generous union (with pruner probability) for later stages
    vc = wv.filter(pl.any_horizontal([rule(**u) for u in UNIONS.values()]))
    for (sc, un), (b, p) in pruned.items():
        vc = vc.join(b.select("scope", "s1", "cand", pl.Series(f"p_{un}", p)), on=["scope", "s1", "cand"], how="left")
    vc.write_parquet(out / "val_candidates.parquet")

    # ---- timing and test-time estimate
    tim = pl.DataFrame(timing + tim_rows, infer_schema_length=None)
    tim.write_csv(out / "timing.csv")
    rate = measured_rates(timing)
    fit = tim.filter(pl.col("method").str.ends_with("_fit_transform") & pl.col("shard").str.starts_with("0/"))  # once per country
    fit_rate = fit["search_s"].sum() / max(fit["pool"].sum() + tr[1].height, 1)
    prune_rate = float(np.mean([x["s_per_row"] for x in tim_rows])) if tim_rows else 0.0  # features + predict per row
    dflt = {r["scope"]: json.loads(r["config"]).get("union") for r in ops.filter(
        (pl.col("country") == "ALL") & (pl.col("target") == DEFAULT_TARGET) & (pl.col("family") == "pruned")).iter_rows(named=True)}
    union_avg = {sc: list_stats(wv.filter((pl.col("scope") == sc) & rule(**UNIONS[dflt.get(sc, "u20")])).select("s1", "cand"),
                                t, s1s, pool_size).filter(pl.col("country") == "ALL")["avg"].item() for sc in a.scopes}
    est = pl.DataFrame(test_est).with_columns(
        (pl.col("forward_pairs") * rate.get("forward", 0.0) * len(FORWARD) / 60).alias("forward_min"),
        (pl.col("reverse_pairs") * rate.get("reverse", 0.0) / 60).alias("reverse_min"),
        ((pl.col("test_s1") + pl.col("test_pool")) * fit_rate / 60).alias("fit_transform_min"),
        (pl.col("test_s1") * pl.col("scope").replace_strict(union_avg, return_dtype=pl.Float64) * prune_rate / 60).alias("pruner_min"),
    ).with_columns(pl.sum_horizontal("forward_min", "reverse_min", "fit_transform_min", "pruner_min").alias("total_min"))
    est.write_csv(out / "test_estimate.csv")
    est_tot = est.group_by("scope").agg(pl.col("^.*_min$").sum()).sort("scope")
    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20

    cols = ["country", "recall", "avg", "median", "p95", "empty_share", "reduction_ratio"]
    report = ["# Blocking evaluation: recall vs candidates per S1 (validation split vs full train pool)\n",
              f"Validation: {s1s.height:,} S1, {t.height:,} true pairs, **{floor:.2f} true matches per S1** (floor for the "
              f"average list size). Pruner trained on {trn_ids.len():,} train-split S1. Validation is split in halves by id "
              f"parity: configs and operating points are chosen on the tune half ({s1s_tune.height:,} S1; curve, Pareto) "
              f"and reported on the report half ({s1s_rep.height:,} S1; operating points, groups).\n"]
    for sc in a.scopes:
        report += [f"# Scope: {sc}\n", "## Operating points\n"]
        for r in ops.filter((pl.col("scope") == sc) & (pl.col("country") == "ALL")).iter_rows(named=True):
            report += [f"### <= {r['target']} candidates per S1: {r['family']} {r['config']}\n",
                       md(ops.filter((pl.col("scope") == sc) & (pl.col("target") == r["target"])).select(cols)) + "\n",
                       md(grp.filter((pl.col("scope") == sc) & (pl.col("target") == r["target"]) & (pl.col("country") == "ALL"))
                          .select("group", "true_pairs", "recall")) + "\n"]
        report += ["## Pareto front (ALL countries)\n",
                   md(front.filter(pl.col("scope") == sc).select("family", "config", "recall", "avg", "median", "p95", "empty_share"), 40) + "\n",
                   "## Best config per family (ALL countries, <= 20 candidates per S1)\n",
                   md(allc.filter((pl.col("scope") == sc) & (pl.col("avg") <= 20)).sort("recall", descending=True)
                      .group_by("family", maintain_order=True).head(3).select("family", "config", "recall", "avg", "median", "p95"), 20) + "\n"]
    report += ["# Timing on validation (seconds)\n", md(tim, 80) + "\n",
               f"Measured rate: forward {rate.get('forward', 0) * 1e9:.2f} ns, reverse {rate.get('reverse', 0) * 1e9:.2f} ns per "
               f"similarity evaluation; searches ran on {'GPU' if blocking.USE_GPU else 'CPU'} ({gpu_info()}). "
               f"Pruner {prune_rate * 1e6:.2f} us per candidate (features + predict).\n",
               "# Test-time estimate (minutes)\n", md(est) + "\n", md(est_tot) + "\n",
               "# Pruner audit\n",
               f"Features ({len(feature_cols())}): {', '.join(feature_cols())}. No country, source or language feature. "
               "Trained only on train-split S1 (asserted disjoint from validation); tau / cap chosen on the tune half.\n",
               md(pl.DataFrame(audit)) + "\n" if audit else "",
               f"Peak host memory of this run: {peak:.1f} GB.\n"]
    (out / "blocking_report.md").write_text("\n".join(report))
    (out / "config.json").write_text(json.dumps({"operating_points": ops.filter(pl.col("country") == "ALL").select(
        "scope", "target", "family", "config", "recall", "avg").to_dicts(), "unions": UNIONS, "default_target": DEFAULT_TARGET,
        "scopes": a.scopes, "pruner_features": feature_cols(),
        "k_max": a.k_max, "m_max": a.m_max, "device": gpu_info(), "peak_gb": peak}, indent=1))
    log("done: " + "; ".join(f"[{r['scope']}] <= {r['target']}: recall {r['recall']:.4f} at {r['avg']:.2f} cands/S1 ({r['family']})"
                             for r in ops.filter(pl.col("country") == "ALL").iter_rows(named=True)))


if __name__ == "__main__":
    main()


In [ ]:
import sys
sys.path.insert(0, '/tmp/src')
sys.argv = ['run_blocking_test_check.py', '--norm-dir', '/kaggle/input', '--pieces', '/kaggle/input', '--model-dir', '/kaggle/input', '--out-dir', '/kaggle/working']

In [ ]:
"""Blocking check on TEST data (no labels), focused on France (test-only country).

Reads test search pieces (run_blocking_eval.py --stage search --side test), the pruners and the chosen operating
point (default ~7 candidates per S1) from a finished evaluation, applies them to test candidates and compares
France with US / India:
  - states: share of S1 and S2/S3 with a detected / inferred (city, dept) / no state; how France gets a state;
  - within-state search groups: number of buckets, the largest groups and their share of all comparisons;
  - candidates per S1 before (starting list) and after the pruner: average, median, p95, empty share, histogram;
  - pruner scores: all candidates and the best candidate per S1; missing-value share of the location features.
Flags anything that suggests French records get worse blocking.

Usage: python src/run_blocking_test_check.py --norm-dir DIR --pieces DIR... --model-dir DIR --out-dir DIR
"""
import argparse
import json
import pickle
import sys
from pathlib import Path

import numpy as np
import polars as pl

try:
    sys.path.insert(0, str(Path(__file__).resolve().parent))
except NameError:  # Kaggle notebook: modules are written to /tmp/src
    sys.path.insert(0, "/tmp/src")
from candidates import add_list_features, features, prune, rule, score  # noqa: E402
from run_blocking_eval import COLS, DEFAULT_TARGET, UNIONS, filled_states, find, log, md, to_wide  # noqa: E402

SLICE_ROWS = 10_000_000  # search rows per slice (bounded memory)
HIST = [(0, 0), (1, 1), (2, 2), (3, 3), (4, 4), (5, 5), (6, 7), (8, 10), (11, 20), (21, 10**9)]


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--norm-dir", default="artifacts/normalised")
    ap.add_argument("--pieces", nargs="+", required=True, help="folders holding piece_test_* files")
    ap.add_argument("--model-dir", required=True, help="folder (searched) with config.json and pruner_*.pkl of an evaluation")
    ap.add_argument("--out-dir", default="artifacts/blocking_test_check")
    a = ap.parse_args()
    out = Path(a.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    norm = lambda sp, k: pl.read_parquet(find(Path(a.norm_dir), f"{sp}_source{k}.parquet", "normalised"), columns=COLS)
    tr = {k: norm("train", k) for k in (1, 2, 3)}
    te = {k: norm("test", k) for k in (1, 2, 3)}
    countries = te[1]["country"].unique().sort().to_list()

    # operating point and pruner from the evaluation (chosen on validation, never on test)
    model_dir = find(Path(a.model_dir), "pruner_state_*.pkl").parent  # the evaluation's output folder (mount paths vary)
    cfg = json.loads((model_dir / "config.json").read_text())
    op = next(o for o in cfg["operating_points"] if o["target"] == DEFAULT_TARGET and o["scope"] == "state")
    opc = json.loads(op["config"])
    assert op["family"] == "pruned", f"default operating point is not a pruned list: {op}"
    union = opc["union"]
    with open(model_dir / f"pruner_state_{union}.pkl", "rb") as f:
        model = pickle.load(f)["model"]
    log(f"operating point <= {DEFAULT_TARGET}: {op['config']} (validation recall {op['recall']:.4f} at {op['avg']:.2f})")

    tef = filled_states(te, tr, countries)  # same maps as the search (train + test records of the country)
    s1_attr = tef[1].select("entity_id", "core_name", "addr_norm", "city", "state")
    pool_attr = pl.concat([tef[2], tef[3]]).select("entity_id", "core_name", "addr_norm", "city", "state")
    del tr, te, tef

    # one country (piece file) at a time, in S1 slices of <= SLICE_ROWS search rows: every pruner input is
    # per S1 (list features, S1 best) or carried in the row (pool best), so slicing by S1 is exact.
    files = sorted({f for d in a.pieces for f in Path(d).rglob("piece_test_*of*.parquet")})
    timing = [t for f in files for t in json.loads(f.with_name(f.stem + "_timing.json").read_text())]
    per_s1, psample, feat = [], [], []
    for f in files:
        country = json.loads(f.with_name(f.stem + "_timing.json").read_text())[0]["country"]
        lz = pl.scan_parquet(f).filter(pl.col("scope") == "state")
        n_parts = max(1, -(-lz.select(pl.len()).collect().item() // SLICE_ROWS))
        for part in range(n_parts):
            sub = lz.filter(pl.col("s1").hash(seed=42) % n_parts == part).collect()
            base = add_list_features(to_wide(sub).filter(rule(**UNIONS[union])))
            del sub
            p = score(model, base, s1_attr, pool_attr)
            base = base.with_columns(pl.Series("p_keep", p))
            kept = prune(base, p, opc["tau"], opc.get("cap"))
            per_s1.append(base.group_by("s1").agg(pl.len().alias("n_start"), pl.col("p_keep").max().alias("p_top"))
                          .join(kept.group_by("s1").len("n_kept"), on="s1", how="left")
                          .with_columns(pl.col("n_kept").fill_null(0), pl.lit(country).alias("country")))
            psample.append(base.select("p_keep").sample(min(base.height, 300_000), seed=42).with_columns(pl.lit(country).alias("country")))
            if part == 0:  # location-feature coverage on a sample of starting-list candidates
                x = features(base.sample(min(base.height, 500_000), seed=42), s1_attr, pool_attr)
                feat.append({"country": country, "rows": x.height, "same_state_missing": (x["f_same_state"] == -1).mean(),
                             "same_city_missing": (x["f_same_city"] == -1).mean(), "same_state_true": (x["f_same_state"] == 1).mean(),
                             "same_city_true": (x["f_same_city"] == 1).mean(), "name_tsr_median": x["f_name_tsr"].median(),
                             "addr_tsr_median": x["f_addr_tsr"].median(), "reverse_found": x["rank_reverse"].is_not_null().mean()})
            log(f"{country} slice {part + 1}/{n_parts}: {base.height:,} starting-list candidates, {kept.height:,} kept")
            del base, kept, p
    per_s1, psample, feat = pl.concat(per_s1), pl.concat(psample), pl.DataFrame(feat)

    # ---- candidates per S1 (S1 with nothing found count as 0)
    queried = {t["country"]: t["s1"] for t in timing if t["method"] == "queried"}
    rows, hist = [], []
    for c in countries:
        for stage, col in (("starting list", "n_start"), ("after pruner", "n_kept")):
            n = per_s1.filter(pl.col("country") == c)[col].to_numpy()
            n = np.concatenate([n, np.zeros(max(queried.get(c, 0) - len(n), 0), dtype=np.int64)])
            if not len(n):
                continue
            rows.append({"country": c, "stage": stage, "s1_queried": queried.get(c, 0), "avg": n.mean(),
                         "median": float(np.median(n)), "p95": float(np.percentile(n, 95)), "empty_share": (n == 0).mean()})
            if stage == "after pruner":
                hist.append({"country": c, **{(f"{lo}" if lo == hi else f"{lo}-{hi}" if hi < 10**9 else f"{lo}+"):
                                              float(((n >= lo) & (n <= hi)).mean()) for lo, hi in HIST}})
    sizes, hist = pl.DataFrame(rows), pl.DataFrame(hist)

    # ---- pruner scores (all candidates: 300k sample per slice; best candidate per S1: exact)
    q = lambda col, qq: pl.col(col).quantile(qq, "nearest")
    scores = psample.group_by("country").agg(*[q("p_keep", x).alias(f"p_all_q{int(x * 100)}") for x in (0.5, 0.9, 0.99)],
                                             (pl.col("p_keep") >= opc["tau"]).mean().alias("share_ge_tau")).join(
        per_s1.group_by("country").agg(*[q("p_top", x).alias(f"p_top1_q{int(x * 100)}") for x in (0.1, 0.25, 0.5)],
                                       (pl.col("p_top") < opc["tau"]).mean().alias("s1_no_cand_ge_tau")), on="country").sort("country")

    # ---- states and search groups (from the search pieces)
    src = pl.DataFrame([t for t in timing if t["method"] == "state_src"]).group_by("country", "side_table", "src").agg(
        pl.col("n").first()).with_columns((pl.col("n") / pl.col("n").sum().over("country", "side_table")).alias("share"))
    src = src.pivot(on="src", index=["country", "side_table"], values="share").fill_null(0.0).sort("country", "side_table")
    bk = pl.DataFrame([t for t in timing if t["method"] == "bucket"]).unique(["country", "bucket"]).sort(
        "country", "pairs_share", descending=[False, True]).select("country", "bucket", "s1", "pool", "pairs_share")
    bsum = pl.DataFrame([t for t in timing if t["method"] == "bucket_summary"]).unique("country").select(
        "country", "n_buckets", "pairs", "s1_total", "pool_total").with_columns(
        (pl.col("pairs") / (pl.col("s1_total") * pl.col("pool_total"))).alias("pairs_vs_whole_country")).sort("country")

    # ---- flags: France vs the mean of US and India
    flags = []
    ref = lambda df, col: df.filter(pl.col("country") != "France")[col].mean()
    fr = lambda df, col: df.filter(pl.col("country") == "France")[col].item() if "France" in df["country"].to_list() else None
    ks = sizes.filter(pl.col("stage") == "after pruner")
    checks = [("candidates per S1 (after pruner)", fr(ks, "avg"), ref(ks, "avg"), lambda f, r: f < 0.7 * r or f > 1.5 * r),
              ("share of S1 with no candidate", fr(ks, "empty_share"), ref(ks, "empty_share"), lambda f, r: f > max(2 * r, r + 0.02)),
              ("median best pruner score per S1", fr(scores, "p_top1_q50"), ref(scores, "p_top1_q50"), lambda f, r: f < r - 0.15),
              ("same-state feature missing", fr(feat, "same_state_missing"), ref(feat, "same_state_missing"), lambda f, r: f > 2 * r + 0.05),
              ("same-city feature missing", fr(feat, "same_city_missing"), ref(feat, "same_city_missing"), lambda f, r: f > 2 * r + 0.05),
              ("comparisons vs whole-country search", fr(bsum, "pairs_vs_whole_country"), ref(bsum, "pairs_vs_whole_country"),
               lambda f, r: f > 3 * r)]
    for name, f, r, bad in checks:
        if f is not None and r is not None:
            flags.append({"check": name, "France": f, "US/India mean": r, "flag": "FLAG" if bad(f, r) else "ok"})
    flags = pl.DataFrame(flags)

    for name, df in (("test_list_sizes", sizes), ("test_list_hist", hist), ("test_pruner_scores", scores),
                     ("test_feature_coverage", feat), ("test_state_sources", src), ("test_buckets", bk),
                     ("test_bucket_summary", bsum), ("test_flags", flags)):
        df.write_csv(out / f"{name}.csv")
    report = [
        "# Blocking on TEST data: France vs US / India (no labels)\n",
        f"Operating point (chosen on validation): <= {DEFAULT_TARGET} candidates per S1, `{op['config']}`; validation recall "
        f"{op['recall']:.4f} at {op['avg']:.2f} candidates per S1. Test S1 queried per country: {queried}.\n",
        "France gets a state in normalisation from its région (18 names, current and pre-2016 régions, hand-written); "
        "départements are a separate field (95 hand-written names) with no département -> région table. Blocking then "
        "infers a missing state from the city, then the département, using maps learned from records that have both "
        "(>= 20 records, >= 90% agreement; train + test records, no labels).\n",
        "## Flags\n", md(flags) + "\n",
        "## Where states come from (share of records)\n", md(src) + "\n",
        "## Within-state search groups\n", md(bsum) + "\n", md(bk) + "\n",
        "## Candidates per S1\n", md(sizes) + "\n", md(hist) + "\n",
        "## Pruner scores\n", md(scores) + "\n",
        "## Pruner feature coverage (sample of starting-list candidates)\n", md(feat) + "\n",
    ]
    (out / "test_check_report.md").write_text("\n".join(report))
    log("flags: " + "; ".join(f"{r['check']}: {r['flag']}" for r in flags.iter_rows(named=True)))


if __name__ == "__main__":
    main()
